# 🕵️ Dia 50 — Imagem de disco bit a bit

**Investigador Cyber 90 Dias** · Fase 4 · Rank Jutsu `B`

> Baseado no *Day 50* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-4-digital-forensics-incident-investigation/day-50.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Na sacola da TI da Serra Azul: o notebook `SA-CX-07` (BitLocker ligado) e um pen drive azul, "AG CENTRO", que estava espetado nele. Você ia abrir o pen drive no Explorer. Rafa puxou o cabo.

Neste notebook você vai:

1. **Montar o "pen drive" EVID-004** do original (FAT16, com um arquivo apagado), em Python puro.
2. **Reimplementar o `dd`** com hash na mesma passada, e provar que imagem = origem.
3. **Simular um setor defeituoso** e ver o que `conv=noerror,sync` faz com `bs=64K` e com `bs=512`.
4. **Medir o que uma cópia de arquivos não alcança.**
5. **Repetir com o pen drive "AG CENTRO"** e gerar a Parte B do formulário.
6. **Jogar o quiz da aquisição.**

🐍 Só biblioteca padrão. O Colab não traz `mkfs.fat`, `mtools`, `dc3dd` nem `ewfacquire`; a célula opcional abaixo instala, mas o notebook não depende dela.


In [ ]:
# (OPCIONAL) Ferramentas reais. Linhas com "!" rodam no shell do Colab; fora dele, são ignoradas.
# Descomente se quiser repetir os comandos do README dentro do Colab.
# !apt-get -qq install -y sleuthkit dc3dd ewf-tools dosfstools mtools > /dev/null
# !dc3dd --version 2>&1 | head -2
print("Célula opcional: nada a fazer se você não descomentou as linhas acima.")


## 1. O "dispositivo"

O original roda `make-evid-004.sh` (com `mkfs.fat` e `mtools`). Aqui um construtor FAT16 em Python faz o mesmo papel: grava setor de boot, as duas FATs, a raiz, os diretórios e os dados, e "apaga" um arquivo do jeito que o `mdel` apaga (troca o 1º byte do nome por `0xE5` e zera a cadeia na FAT, **sem** apagar os bytes).

> Diferenças honestas: o nosso volume tem 4 MiB (o original, 16 MiB) e clusters de 512 bytes. A estrutura é a mesma; o tamanho é menor para caber no notebook. O seu hash não vai bater com o de quem rodou o script do original, e o próprio original avisa que os hashes variam entre alunos.


In [ ]:
# ------------------------------------------------------------
# fat16lab — um "pen drive" FAT16 de brinquedo, em Python puro.
# Constrói uma imagem FAT16 pequena (4 MiB) e lê essa imagem
# como The Sleuth Kit faria (fsstat, fls, istat, icat -r).
# Só biblioteca padrão. Tudo determinístico: mesma entrada,
# mesmos bytes, mesmo SHA-256.
# ------------------------------------------------------------
import struct                          # empacota números em bytes (little-endian, como o FAT)
from datetime import datetime          # datas dos arquivos

BPS = 512            # bytes por setor
SPC = 1              # setores por cluster (cluster = 512 bytes, pequeno de propósito)
RESERVADOS = 1       # só o setor de boot
NFATS = 2            # FAT e cópia da FAT, como em todo FAT16
ENTRADAS_RAIZ = 512  # a raiz do FAT16 tem tamanho fixo
TOTAL_SETORES = 8192 # 8192 x 512 = 4 MiB
SETORES_FAT = 32     # 32 x 512 / 2 bytes = 8192 entradas, cobre todos os clusters


def _data_fat(dt):
    # FAT guarda a data em 16 bits: ano desde 1980 (7), mês (4), dia (5)
    return ((dt.year - 1980) << 9) | (dt.month << 5) | dt.day


def _hora_fat(dt):
    # e a hora em 16 bits: hora (5), minuto (6), segundos/2 (5). Sem fuso!
    return (dt.hour << 11) | (dt.minute << 5) | (dt.second // 2)


def _nome83(nome):
    # "VENDORS.CSV" -> b"VENDORS CSV" (8 + 3, preenchido com espaços)
    base, _, ext = nome.upper().partition(".")
    return (base[:8].ljust(8) + ext[:3].ljust(3)).encode("ascii")


def _entrada(nome11, atributo, cluster, tamanho, criado, modificado):
    # Uma entrada de diretório tem 32 bytes. A ordem dos campos é a do FAT.
    return struct.pack(
        "<11sBBBHHHHHHHI",
        nome11, atributo, 0,
        0,                                         # décimos de segundo da criação
        _hora_fat(criado), _data_fat(criado),      # criação
        _data_fat(modificado),                     # último acesso: SÓ data
        0,                                         # cluster alto (zero no FAT16)
        _hora_fat(modificado), _data_fat(modificado),  # última escrita
        cluster, tamanho)


def construir_fat16(rotulo, id_volume, pastas, arquivos, apagar=(), criado_em=None):
    """
    rotulo: texto de até 11 letras; id_volume: inteiro de 32 bits.
    pastas: {"FINANCE": datetime}; arquivos: [("FINANCE/VENDORS.CSV", bytes, datetime_modificacao)].
    apagar: caminhos que serão "deletados" como o mdel faz (0xE5 + cadeia zerada).
    criado_em: hora de criação gravada em todos (o mtools grava a hora da cópia).
    Devolve bytearray com a imagem inteira.
    """
    img = bytearray(TOTAL_SETORES * BPS)                    # o "dispositivo", todo em zero
    inicio_fat = RESERVADOS * BPS
    inicio_raiz = inicio_fat + NFATS * SETORES_FAT * BPS
    inicio_dados = inicio_raiz + ENTRADAS_RAIZ * 32
    tam_cluster = BPS * SPC

    # --- setor de boot (BPB) ---
    boot = struct.pack("<3s8sHBHBHHBHHHII", b"\xEB\x3C\x90", b"PYLAB1.0",
                       BPS, SPC, RESERVADOS, NFATS, ENTRADAS_RAIZ, TOTAL_SETORES,
                       0xF8, SETORES_FAT, 32, 64, 0, 0)
    boot += struct.pack("<BBBI11s8s", 0x80, 0, 0x29, id_volume,
                        rotulo.upper().ljust(11)[:11].encode("ascii"), b"FAT16   ")
    img[0:len(boot)] = boot
    img[510:512] = b"\x55\xAA"                              # assinatura de setor de boot

    fat = [0] * (SETORES_FAT * BPS // 2)
    fat[0], fat[1] = 0xFFF8, 0xFFFF                         # entradas reservadas
    proximo = [2]                                           # primeiro cluster de dados é o 2

    def alocar(n):
        # aloca n clusters contíguos e encadeia na FAT
        primeiro = proximo[0]
        for i in range(n):
            c = primeiro + i
            fat[c] = 0xFFFF if i == n - 1 else c + 1
        proximo[0] += n
        return primeiro, list(range(primeiro, primeiro + n))

    def offset_cluster(c):
        return inicio_dados + (c - 2) * tam_cluster

    criado_em = criado_em or datetime(2026, 1, 1, 0, 0, 0)
    raiz = [_entrada(rotulo.upper().ljust(11)[:11].encode("ascii"), 0x08, 0, 0, criado_em, criado_em)]
    subdirs = {}                                            # nome -> (cluster, lista de entradas)
    for nome, quando in pastas.items():
        c, _ = alocar(1)
        subdirs[nome] = (c, [
            _entrada(b".          ", 0x10, c, 0, quando, quando),
            _entrada(b"..         ", 0x10, 0, 0, quando, quando)])
        raiz.append(_entrada(_nome83(nome), 0x10, c, 0, quando, quando))

    clusters_de = {}
    for item in arquivos:
        caminho, conteudo, mtime = item[:3]
        ctime = item[3] if len(item) > 3 else criado_em     # 4º campo opcional: hora de criação
        n = max(1, -(-len(conteudo) // tam_cluster))        # divisão arredondando para cima
        c, lista = alocar(n)
        clusters_de[caminho] = lista
        for i, cl in enumerate(lista):                      # grava o conteúdo cluster a cluster
            pedaco = conteudo[i * tam_cluster:(i + 1) * tam_cluster]
            o = offset_cluster(cl)
            img[o:o + len(pedaco)] = pedaco
        e = _entrada(_nome83(caminho.split("/")[-1]), 0x20, c, len(conteudo), ctime, mtime)
        if "/" in caminho:
            subdirs[caminho.split("/")[0]][1].append(e)
        else:
            raiz.append(e)

    def marcar_apagado(lista_entradas, nome11):
        for i, e in enumerate(lista_entradas):
            if e[:11] == nome11:
                lista_entradas[i] = b"\xE5" + e[1:]         # o "apagar" do FAT: 1º byte vira 0xE5

    for caminho in apagar:
        nome11 = _nome83(caminho.split("/")[-1])
        alvo = subdirs[caminho.split("/")[0]][1] if "/" in caminho else raiz
        marcar_apagado(alvo, nome11)
        for cl in clusters_de[caminho]:
            fat[cl] = 0                                     # cadeia zerada; os BYTES ficam lá

    img[inicio_raiz:inicio_raiz + 32 * len(raiz)] = b"".join(raiz)
    for nome, (c, entradas) in subdirs.items():
        o = offset_cluster(c)
        img[o:o + 32 * len(entradas)] = b"".join(entradas)
    fat_bytes = struct.pack("<%dH" % len(fat), *fat)
    for k in range(NFATS):                                  # FAT1 e FAT2 idênticas
        o = inicio_fat + k * SETORES_FAT * BPS
        img[o:o + len(fat_bytes)] = fat_bytes
    return img


In [ ]:
# ------------------------------------------------------------
# Os dois "pen drives" sintéticos do dia
# ------------------------------------------------------------
from datetime import datetime

# EVID-004 do LAB-P4: os mesmos arquivos e datas do make-evid-004.sh do original.
PNG_1x1 = (b"\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00\x01\x00\x00\x00\x01\x08\x06\x00\x00\x00"
           b"\x1f\x15\xc4\x89\x00\x00\x00\rIDATx\x9cc\xf8\x0f\x00\x00\x01\x01\x00\x05\x18\xd8N"
           b"\x00\x00\x00\x00IEND\xaeB`\x82")          # o PNG de 1x1 escondido atrás de .TXT
EVID_004 = dict(
    rotulo="LABUSB04", id_volume=0x4C414234,
    pastas={"FINANCE": datetime(2026, 3, 13, 9, 30, 0)},
    arquivos=[
        ("README.TXT", b"SYNTHETIC LAB MEDIA - case LAB-P4 - Example Fabrication Co. (fictional)\n",
         datetime(2026, 3, 10, 14, 5, 0)),
        ("FINANCE/VENDORS.CSV", b"# SYNTHETIC - fictional vendors for a training image\n"
         b"vendor_id,name,iban_last4,approved_by\nV-1001,Acme Test Supplies,0042,fin.mgr01\n"
         b"V-1002,Contoso Sample Metals,7719,fin.mgr01\nV-1003,Placeholder Logistics,3308,acct.lead02\n",
         datetime(2026, 3, 11, 16, 20, 0)),
        ("FINANCE/MEETING.TXT", b"SYNTHETIC - Q1 close checklist\n1. Reconcile vendor ledger\n"
         b"2. Export invoices for audit\n", datetime(2026, 3, 12, 9, 45, 0)),
        ("FINANCE/SCAN0001.TXT", PNG_1x1, datetime(2026, 3, 12, 10, 2, 0)),
    ],
    apagar=["FINANCE/VENDORS.CSV"],
    criado_em=datetime(2026, 3, 13, 9, 30, 0),       # no original, a hora em que o script rodou
)

# ➕ EVID-SA-004: o pen drive "AG CENTRO" da Serra Azul (fictício). Horas em BRASÍLIA (FAT não tem fuso).
ZIP_FALSO = (b"PK\x03\x04" + b"\x00" * 26 +
             b"OPERACAO ISCA - ARQUIVO SINTETICO. Representa o 'Modulo de Seguranca' do IscaKit. "
             b"Nao contem codigo. Nada aqui executa.\n" * 6)
EVID_SA_004 = dict(
    rotulo="AG CENTRO", id_volume=0x53413034,
    pastas={"TARIFAS": datetime(2026, 9, 1, 9, 12, 4)},
    arquivos=[
        # (caminho, conteúdo, modificado, criado). Copiar no Windows preserva o "modificado" e
        # grava como "criado" a hora da cópia: é por isso que o ZIP tem criado DEPOIS de modificado.
        ("LEIAME.TXT", b"Pen drive da agencia Centro - uso interno - Cooperativa Serra Azul (ficticia)\n",
         datetime(2026, 9, 1, 9, 12, 0), datetime(2026, 9, 1, 9, 12, 0)),
        ("TARIFAS/TAB2026.CSV", b"servico,valor\nTED,0.00\nDOC,0.00\nSaque extra,2.50\n",
         datetime(2026, 9, 30, 16, 40, 12), datetime(2026, 9, 30, 16, 40, 12)),
        ("TARIFAS/AVISO.TXT", b"Atualizar a tabela no caixa 2 toda segunda-feira.\n",
         datetime(2026, 10, 2, 11, 5, 30), datetime(2026, 10, 2, 11, 5, 30)),
        ("MODULO_S.ZIP", ZIP_FALSO, datetime(2026, 10, 5, 8, 23, 14), datetime(2026, 10, 5, 8, 35, 10)),
    ],
    apagar=["MODULO_S.ZIP"],
    criado_em=datetime(2026, 10, 5, 8, 35, 10),
)
print("Receitas prontas: EVID_004 (LAB-P4) e EVID_SA_004 (pen drive AG CENTRO).")


In [ ]:
import hashlib
from pathlib import Path

LAB = Path("lab-p4"); (LAB / "evidence").mkdir(parents=True, exist_ok=True)
(LAB / "work").mkdir(exist_ok=True); (LAB / "notes").mkdir(exist_ok=True)

origem = LAB / "evidence" / "evid-004-usb.dd"
if origem.exists():
    origem.chmod(0o644)                    # se rodar de novo, devolve a escrita para recriar
origem.write_bytes(construir_fat16(**EVID_004))
origem.chmod(0o444)                        # chmod a-w, como no passo 1 do original

def sha256_arquivo(p, bs=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while bloco := f.read(bs):
            h.update(bloco)
    return h.hexdigest()

HASH_ORIGEM = sha256_arquivo(origem)
print(f"{HASH_ORIGEM}  {origem.name}")
print(f"Tamanho: {origem.stat().st_size} bytes = {origem.stat().st_size // 512} setores de 512")
print("Built evid-004-usb.dd. Record the SHA-256 above on your chain-of-custody form before analysis.")


## 2. O `dd`, em Python

`dd if=ORIGEM of=DESTINO bs=64K conv=noerror,sync` faz quatro coisas: lê blocos de `bs` bytes, grava cada bloco, **continua** se a leitura falhar (`noerror`) e **preenche com zeros** o bloco que falhou (`sync`), para que tudo o que vem depois continue no *offset* certo.

A nossa função faz o mesmo, calcula o SHA-256 enquanto copia (como o `dc3dd`) e devolve um registro de aquisição.


In [ ]:
import time
from datetime import datetime, timezone

def agora():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def dd(origem, destino, bs=64 * 1024, setores_ruins=(), noerror=True, sync=True):
    """
    Copia 'origem' para 'destino' em blocos de 'bs' bytes.
    setores_ruins: números de setor (512 bytes) que vamos FINGIR que não se deixam ler.
    Devolve um dicionário com tudo o que vai para a Parte B do formulário.
    """
    total = Path(origem).stat().st_size
    h_entrada = hashlib.sha256()                 # o que conseguimos ler (como o "input hash" do dc3dd)
    erros = []                                   # (offset, tamanho) dos blocos que falharam
    inicio = agora()
    with open(origem, "rb") as src, open(destino, "wb") as dst:
        offset = 0
        while offset < total:
            bloco = src.read(bs)
            # Algum setor ruim cai dentro deste bloco? (setor s ocupa os bytes s*512 .. s*512+511)
            ruim = any(offset <= s * 512 < offset + len(bloco) for s in setores_ruins)
            if ruim:
                erros.append((offset, len(bloco)))
                if not noerror:
                    raise IOError(f"dd: erro de leitura no offset {offset} (sem noerror, o dd para aqui)")
                bloco = b"\x00" * len(bloco) if sync else b""    # sync: o bloco INTEIRO vira zero
            h_entrada.update(bloco)
            dst.write(bloco)
            offset += bs
    return dict(origem=str(origem), destino=str(destino), bs=bs, inicio=inicio, fim=agora(),
                hash_lido=h_entrada.hexdigest(), hash_imagem=sha256_arquivo(destino),
                erros=erros, bytes_zerados=sum(t for _, t in erros))

img_raw = LAB / "work" / "evid-004.raw"
r = dd(origem, img_raw, bs=64 * 1024)
print(f"origem : {HASH_ORIGEM}")
print(f"imagem : {r['hash_imagem']}")
print("confere:", "SIM ✅" if r["hash_imagem"] == HASH_ORIGEM else "NÃO ❌")


### Três imagens, um hash de origem

O original gera três imagens (dd, dc3dd e E01) e compara cada uma com a origem. Aqui: `bs=64K`, `bs=512` e uma terceira com `bs=4096`, todas sem setor ruim. Tamanho de bloco **não** muda o resultado quando a leitura é perfeita.


In [ ]:
REGISTROS = {}
for nome, bs in [("evid-004.raw", 64 * 1024), ("evid-004.bs512.raw", 512), ("evid-004.bs4k.raw", 4096)]:
    REGISTROS[nome] = dd(origem, LAB / "work" / nome, bs=bs)
    ok = REGISTROS[nome]["hash_imagem"] == HASH_ORIGEM
    print(f"{nome:22} bs={bs:>6}  {REGISTROS[nome]['hash_imagem'][:20]}…  confere: {'sim' if ok else 'NÃO'}")


## 3. 💥 Um setor defeituoso

Agora fingimos que o setor 100 (bytes 51 200 a 51 711) não se deixa ler. Repare:

- com `bs=64K`, o `sync` zera o **bloco inteiro** de 64 KiB que contém o setor;
- com `bs=512`, zera só o setor.

É a frase do original: *"com um `bs` grande, um único setor defeituoso zera o bloco inteiro"*.


In [ ]:
RUINS = [100]
for bs in (64 * 1024, 512):
    rr = dd(origem, LAB / "work" / f"ruim-bs{bs}.raw", bs=bs, setores_ruins=RUINS)
    print(f"bs={bs:>6}: blocos com erro {rr['erros']} -> {rr['bytes_zerados']:>6} bytes viraram zero; "
          f"hash bate com a origem? {'sim' if rr['hash_imagem'] == HASH_ORIGEM else 'NÃO'}")

# E sem noerror? O dd simplesmente para.
try:
    dd(origem, LAB / "work" / "ruim-sem-noerror.raw", bs=512, setores_ruins=RUINS, noerror=False)
except IOError as e:
    print("Sem noerror:", e)


> ⚠️ **A lição do passo 8:** com o setor ruim, o hash da imagem **não** bate com o da origem. O que você escreve? O que observou (setor 100 ilegível, 512 bytes zerados, ferramenta e `bs`), os dois hashes, e que vai adquirir de novo (no mundo real, com `ddrescue`, que tenta de novo os setores difíceis). Você **não** escolhe o hash que prefere.

## 4. O que a cópia de arquivos não vê

Uma "cópia de arquivos" pega só o conteúdo dos arquivos vivos. Quanto do dispositivo fica de fora? E o arquivo apagado `VENDORS.CSV` está no dispositivo?


In [ ]:
# ------------------------------------------------------------
# Um "mini Sleuth Kit" para FAT16, em Python puro.
# fsstat -> ler_bpb | fls -r -p -> listar | istat -> entrada | icat -r -> recuperar
# Só LÊ a imagem (bytes). Nunca escreve nela.
# ------------------------------------------------------------
import struct
from datetime import datetime


def ler_bpb(img):
    # O setor de boot diz a geometria do volume: é o que o fsstat imprime.
    (bps, spc, res, nfats, nraiz, total, _mid, spf) = struct.unpack_from("<HBHBHHBH", img, 11)
    id_vol, = struct.unpack_from("<I", img, 39)
    rotulo = img[43:54].decode("ascii").strip()
    tipo = img[54:62].decode("ascii").strip()
    inicio_fat = res * bps
    inicio_raiz = inicio_fat + nfats * spf * bps
    inicio_dados = inicio_raiz + nraiz * 32
    return dict(bps=bps, spc=spc, nfats=nfats, nraiz=nraiz, total=total, spf=spf,
                id_volume=id_vol, rotulo=rotulo, tipo=tipo, inicio_fat=inicio_fat,
                inicio_raiz=inicio_raiz, inicio_dados=inicio_dados,
                tam_cluster=bps * spc)


def _dt(data, hora=0):
    # Decodifica data/hora do FAT. Sem fuso: quem decide o fuso é VOCÊ (o -z do fls).
    if data == 0:
        return None
    ano, mes, dia = 1980 + (data >> 9), (data >> 5) & 15, data & 31
    h, m, s = hora >> 11, (hora >> 5) & 63, (hora & 31) * 2
    return datetime(ano, mes, dia, h, m, s)


def _entradas(img, bpb, offset, quantidade):
    # Lê entradas de 32 bytes a partir de um offset; para no primeiro byte 0x00 (fim).
    for i in range(quantidade):
        o = offset + i * 32
        e = img[o:o + 32]
        if e[0] == 0x00:
            break
        nome = e[0:8].decode("latin-1").rstrip()
        ext = e[8:11].decode("latin-1").rstrip()
        atributo = e[11]
        (_, _, hc, dc, da, _, hm, dm, cluster, tamanho) = struct.unpack_from("<BBHHHHHHHI", e, 12)
        apagado = e[0] == 0xE5
        if apagado:
            nome = "_" + nome[1:]          # o TSK mostra o 1º caractere perdido como "_"
        yield dict(nome=nome + ("." + ext if ext else ""), atributo=atributo,
                   apagado=apagado, cluster=cluster, tamanho=tamanho,
                   criado=_dt(dc, hc), acesso=_dt(da), modificado=_dt(dm, hm), offset=o)


def listar(img):
    """Equivale a fls -r -p: devolve todas as entradas, inclusive apagadas."""
    bpb = ler_bpb(img)
    saida = []

    def visitar(offset, quantidade, prefixo):
        for e in _entradas(img, bpb, offset, quantidade):
            if e["atributo"] & 0x08 or e["nome"] in (".", ".."):
                continue                    # rótulo do volume e atalhos . e ..
            e["caminho"] = prefixo + e["nome"]
            # "inode" no estilo TSK para FAT: posição da entrada em blocos de 32 bytes
            e["inode"] = e["offset"] // 32
            saida.append(e)
            if e["atributo"] & 0x10 and not e["apagado"]:
                o = bpb["inicio_dados"] + (e["cluster"] - 2) * bpb["tam_cluster"]
                visitar(o, bpb["tam_cluster"] // 32, e["caminho"] + "/")

    visitar(bpb["inicio_raiz"], bpb["nraiz"], "")
    return saida


def recuperar(img, entrada):
    """Equivale a icat -r: lê 'tamanho' bytes a partir do 1º cluster, supondo contiguidade."""
    bpb = ler_bpb(img)
    o = bpb["inicio_dados"] + (entrada["cluster"] - 2) * bpb["tam_cluster"]
    return bytes(img[o:o + entrada["tamanho"]])


In [ ]:
dados = origem.read_bytes()
vivos = [e for e in listar(dados) if not e["apagado"] and not e["atributo"] & 0x10]
bytes_copia = sum(e["tamanho"] for e in vivos)
print("Cópia de arquivos (Explorer/cp) levaria:")
for e in vivos:
    print(f"   {e['caminho']:24} {e['tamanho']:>5} bytes")
print(f"   total: {bytes_copia} bytes de {len(dados)} ({100 * bytes_copia / len(dados):.4f}% do dispositivo)")

apagado = next(e for e in listar(dados) if e["apagado"])
print(f"\nNa imagem, mas fora da cópia: {apagado['caminho']} (apagado), {apagado['tamanho']} bytes.")
print("Os bytes dele ainda estão no dispositivo?", b"Acme Test Supplies" in dados)


## 5. ➕ O pen drive "AG CENTRO" (EVID-SA-004)

Mesmo processo, outro dispositivo. Depois, a Parte B do formulário de custódia, gerada a partir do registro da aquisição (é assim que se evita erro de digitação num hash de 64 caracteres).


In [ ]:
#@title Aquisição do EVID-SA-004 { display-mode: "form" }
examinador = "Você, analista júnior, Vigia"  #@param {type:"string"}
bloqueador = "Software: arquivo somente leitura (laboratório); no real, bloqueador USB de hardware"  #@param {type:"string"}
bs_escolhido = 512  #@param {type:"integer"}

ISCA = Path("caso-isca"); (ISCA / "evidence").mkdir(parents=True, exist_ok=True)
(ISCA / "work").mkdir(exist_ok=True); (ISCA / "notes").mkdir(exist_ok=True)
origem_sa = ISCA / "evidence" / "evid-sa-004-usb.dd"
if origem_sa.exists():
    origem_sa.chmod(0o644)
origem_sa.write_bytes(construir_fat16(**EVID_SA_004))
origem_sa.chmod(0o444)

hash_origem_sa = sha256_arquivo(origem_sa)                  # passo: hash da origem ANTES
reg = dd(origem_sa, ISCA / "work" / "evid-sa-004.raw", bs=bs_escolhido)
hash_origem_depois = sha256_arquivo(origem_sa)              # e DEPOIS: a origem não pode ter mudado
confere = reg["hash_imagem"] == hash_origem_sa == hash_origem_depois

parte_b = f"""## Parte B: registro de aquisição — EVID-SA-004 (pen drive "AG CENTRO", fictício)

| Campo | Registro |
|---|---|
| Bloqueador de escrita | {bloqueador} |
| Ferramenta e versão | dd em Python (notebook do Dia 50), Python {__import__('sys').version.split()[0]} |
| Comando exato | dd(origem, destino, bs={bs_escolhido}, noerror=True, sync=True) |
| Hash da origem (antes), SHA-256 | {hash_origem_sa} |
| Hash da origem (depois), SHA-256 | {hash_origem_depois} |
| Arquivo de imagem e formato | evid-sa-004.raw (raw/dd) |
| Hash da imagem, SHA-256 | {reg['hash_imagem']} |
| Os hashes conferem? | {'sim' if confere else 'NÃO: registrar, não escolher, adquirir de novo'} |
| Setores defeituosos / erros de leitura | {reg['erros'] or 'nenhum'} |
| Início e fim (UTC) | {reg['inicio']} — {reg['fim']} |
| Assinatura do examinador | {examinador} |
"""
destino = ISCA / "notes" / "parte-b-EVID-SA-004.md"
destino.write_text(parte_b, encoding="utf-8")
print(parte_b)
print("💾", destino)


> 🔎 **Pista para amanhã:** o pen drive "AG CENTRO" também tem um arquivo apagado. O Dia 51 vai encontrá-lo, recuperá-lo e colocá-lo numa linha do tempo, e vai mostrar por que a hora dele **não** está em UTC.

## 6. 🎮 Quiz da aquisição


In [ ]:
MODO_JOGO = True   # False = só o gabarito

PERGUNTAS = [
    ("O que conv=noerror faz no dd?",
     ["Ignora o hash", "Continua depois de um erro de leitura", "Comprime a imagem"], 2,
     "Sem ele, o dd para no primeiro setor ilegível."),
    ("E o sync, junto com noerror?",
     ["Sincroniza o relógio", "Preenche com zeros o bloco que falhou", "Grava em E01"], 2,
     "Mantém os offsets corretos depois do erro; com bs grande, zera muito mais que o setor."),
    ("Qual formato guarda dados do caso e um hash embutido?",
     ["raw/dd", "E01 (EWF)", "ZIP"], 2, "O ewfverify recalcula e compara com o hash embutido."),
    ("O hash da origem depois da imagem não bate com o de antes. Você...",
     ["fica com o que bate com a imagem", "registra os dois, investiga e adquire de novo", "apaga o primeiro"], 2,
     "Nunca escolha o hash que convém. Pen drives e SSDs remapeiam blocos."),
    ("Bloqueio por software (blockdev --setro) comparado com hardware é...",
     ["evidência mais fraca, e precisa constar nas notas", "igual", "proibido"], 1,
     "Vale, mas o relatório tem que dizer qual foi usado."),
    ("➕ Notebook com BitLocker: a imagem do disco inteiro, sem a chave, serve para quê?",
     ["Nada, apague", "É a cópia fiel do dispositivo; a análise precisa da chave ou do volume aberto",
      "Já dá para ler os arquivos"], 2,
     "Faça as duas se puder; registre quem deu a chave, sem escrevê-la."),
]

acertos = 0
for i, (perg, ops, certa, porque) in enumerate(PERGUNTAS, 1):
    print(f"\n[{i}/{len(PERGUNTAS)}] {perg}")
    for k, o in enumerate(ops, 1):
        print(f"   {k}. {o}")
    if MODO_JOGO:
        r = input("   Resposta (número): ").strip()
        ok = r == str(certa)
        acertos += ok
        print("   ✅" if ok else "   ❌", end=" ")
    print(f"Certa: {certa}. {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(PERGUNTAS)}")


In [ ]:
# No Colab, baixa a Parte B para o seu fork.
try:
    from google.colab import files
    files.download("caso-isca/notes/parte-b-EVID-SA-004.md")
except ImportError:
    print("(Fora do Colab: o arquivo ficou em caso-isca/notes/)")


## ✅ Antes de fechar o Dia 50

- [ ] três imagens do EVID-004, todas iguais à origem
- [ ] você explica `noerror` e `sync`, e por que `bs=64K` zerou 64 KiB
- [ ] o parágrafo da divergência: registrar, adquirir de novo, não escolher
- [ ] Parte B do EVID-SA-004 gerada
- [ ] plano de aquisição do `SA-CX-07` (BitLocker) escrito no README

**Amanhã (Dia 51):** você abre a imagem por baixo dos arquivos, como o The Sleuth Kit: `fsstat`, `fls`, `istat`, `icat -r` e uma linha do tempo.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>
